# Neural Network from Scratch - XOR Problem

This notebook demonstrates a complete feedforward neural network implementation using only NumPy.

**From:** [NLP Knowledge Base - Neural Networks Guide](https://github.com/rahulbasu-dev/nlp-kb)

## Install Dependencies

In [ ]:
import numpy as np
print(f"NumPy version: {np.__version__}")

## Neural Network Class Definition

In [ ]:
class NeuralNetwork:
    """
    Feedforward neural network with backpropagation.

    Example:
        nn = NeuralNetwork([2, 4, 1], learning_rate=0.5)
        nn.train(X, y, epochs=10000)
        predictions = nn.predict(X)
    """

    def __init__(self, layer_sizes, learning_rate=0.01):
        """
        Initialize network.

        Args:
            layer_sizes: List [input_dim, hidden1, ..., output_dim]
            learning_rate: Gradient descent step size
        """
        self.layer_sizes = layer_sizes
        self.learning_rate = learning_rate
        self.weights = []
        self.biases = []
        self._initialize_parameters()

    def _initialize_parameters(self):
        """Xavier initialization for weights."""
        for i in range(len(self.layer_sizes) - 1):
            n_in = self.layer_sizes[i]
            n_out = self.layer_sizes[i + 1]

            # Xavier initialization
            W = np.random.randn(n_in, n_out) * np.sqrt(2.0 / n_in)
            b = np.zeros((1, n_out))

            self.weights.append(W)
            self.biases.append(b)

    def sigmoid(self, z):
        """Sigmoid activation: σ(z) = 1/(1+e^(-z))"""
        return 1 / (1 + np.exp(-np.clip(z, -500, 500)))

    def sigmoid_derivative(self, a):
        """Derivative: σ'(z) = σ(z)(1-σ(z))"""
        return a * (1 - a)

    def forward(self, X):
        """
        Forward propagation.

        Returns:
            activations: List of activations [a0, a1, ..., aL]
            z_values: List of pre-activation values
        """
        activations = [X]
        z_values = []

        for W, b in zip(self.weights, self.biases):
            z = np.dot(activations[-1], W) + b
            a = self.sigmoid(z)
            z_values.append(z)
            activations.append(a)

        return activations, z_values

    def compute_loss(self, y_true, y_pred):
        """Mean squared error loss."""
        return np.mean((y_pred - y_true) ** 2)

    def backward(self, X, y, activations, z_values):
        """
        Backpropagation to compute gradients.

        Returns:
            weight_grads: Gradients for each weight matrix
            bias_grads: Gradients for each bias vector
        """
        m = X.shape[0]  # Number of samples
        num_layers = len(self.weights)

        # Output layer error
        delta = (activations[-1] - y) * self.sigmoid_derivative(activations[-1])
        deltas = [delta]

        # Backpropagate through hidden layers
        for l in range(num_layers - 2, -1, -1):
            delta = np.dot(deltas[0], self.weights[l+1].T) * \
                    self.sigmoid_derivative(activations[l+1])
            deltas.insert(0, delta)

        # Compute weight and bias gradients
        weight_grads = []
        bias_grads = []

        for l in range(num_layers):
            dW = np.dot(activations[l].T, deltas[l]) / m
            db = np.sum(deltas[l], axis=0, keepdims=True) / m
            weight_grads.append(dW)
            bias_grads.append(db)

        return weight_grads, bias_grads

    def update_parameters(self, weight_grads, bias_grads):
        """Gradient descent update."""
        for i in range(len(self.weights)):
            self.weights[i] -= self.learning_rate * weight_grads[i]
            self.biases[i] -= self.learning_rate * bias_grads[i]

    def train(self, X, y, epochs=1000, verbose=True):
        """
        Train the neural network.

        Args:
            X: Input data [n_samples, n_features]
            y: Target labels [n_samples, n_outputs]
            epochs: Number of training iterations
            verbose: Print progress

        Returns:
            losses: List of loss values per epoch
        """
        losses = []

        for epoch in range(epochs):
            # Forward pass
            activations, z_values = self.forward(X)

            # Compute loss
            loss = self.compute_loss(y, activations[-1])
            losses.append(loss)

            # Backward pass
            weight_grads, bias_grads = self.backward(X, y, activations, z_values)

            # Update weights
            self.update_parameters(weight_grads, bias_grads)

            # Print progress
            if verbose and (epoch % (epochs // 10) == 0 or epoch == epochs - 1):
                print(f"Epoch {epoch:4d}: Loss = {loss:.6f}")

        return losses

    def predict(self, X):
        """Make predictions on new data."""
        activations, _ = self.forward(X)
        return activations[-1]

## XOR Problem Demonstration

The XOR (exclusive OR) problem is a classic test for neural networks because it's not linearly separable.

A single perceptron cannot solve XOR, but a network with one hidden layer can!

In [ ]:
print("=" * 60)
print("NEURAL NETWORK DEMONSTRATION: XOR PROBLEM")
print("=" * 60)

# XOR dataset (linearly inseparable)
X = np.array([[0, 0],
              [0, 1],
              [1, 0],
              [1, 1]])

y = np.array([[0],
              [1],
              [1],
              [0]])

print("\nDataset:")
print("Input  | Target")
print("-------|-------")
for i in range(len(X)):
    print(f"{X[i]}  | {y[i][0]}")

## Train the Network

In [ ]:
# Create and train network
print("\nInitializing network: [2, 4, 1]")
print("(2 inputs → 4 hidden → 1 output)\n")

nn = NeuralNetwork([2, 4, 1], learning_rate=0.5)
losses = nn.train(X, y, epochs=10000, verbose=True)

## Test the Trained Network

In [ ]:
print("\n" + "=" * 60)
print("FINAL PREDICTIONS")
print("=" * 60)
predictions = nn.predict(X)

print("\nInput  | Target | Prediction | Rounded")
print("-------|--------|------------|--------")
for i in range(len(X)):
    pred = predictions[i][0]
    print(f"{X[i]}  | {y[i][0]:6.0f} | {pred:10.4f} | {round(pred):7.0f}")

print("\n✓ Network successfully learned XOR function!")

## Visualize Training Loss

In [ ]:
import matplotlib.pyplot as plt

plt.figure(figsize=(10, 6))
plt.plot(losses, linewidth=2)
plt.xlabel('Epoch', fontsize=12)
plt.ylabel('Loss (MSE)', fontsize=12)
plt.title('Training Loss Over Time', fontsize=14, fontweight='bold')
plt.grid(True, alpha=0.3)
plt.yscale('log')
plt.tight_layout()
plt.show()

print(f"Final loss: {losses[-1]:.6f}")

## 🎯 Experiment!

Try modifying the code:

1. **Different architecture:** Change `[2, 4, 1]` to `[2, 8, 1]` or `[2, 4, 4, 1]`
2. **Learning rate:** Try `0.1`, `1.0`, or `2.0` (too high will fail!)
3. **More epochs:** Increase to `20000` for potentially better results
4. **Different problem:** Try AND or OR (easier than XOR)

```python
# AND problem
y_and = np.array([[0], [0], [0], [1]])

# OR problem  
y_or = np.array([[0], [1], [1], [1]])
```